## Stage 2 - Gather data -> a pickled `datacol`

Sample for `edmft-aly-skill` Section 3. Copy this into `$PLOT_WORKDIR` as `DataCol.ipynb`
and adapt the marked `<...>` placeholders. It walks one converged eDMFT study into a
`datacol` and pickles it; Stage 3 (`Plot.ipynb`) reloads that pickle.

**The model.** A mission spans one study and may hold **several cases / temperatures** (all
the `edmft-T<T>/` run folders under one parent). **Default to a single mission** pointed at
that parent: `pull()` walks the whole tree and tags each file by its path, so T-points /
cases stay distinguished within the one mission. Add a second mission only for a genuinely
separate study. Behind each folder is a *batch* of calculations the next parameter batch
overwrites in place, so the folder is transient and the `datacol` pickle is the **durable
record**. Working style: the **objects are the state** - build, `save_pkl`, then comment the
build block out; to add a case or re-collect a new batch later, reload and *append* /
re-`pull` - never rebuild from scratch.

### Step 0 - Imports

`data.py` / `quickplot.py` must be importable. Point `sys.path` at the folder that
holds them (this skill's `reference/` folder, or wherever you copied them on the
server). The libs are Python-2 compatible.

In [ ]:
import sys
sys.path.append('<dir with data.py / quickplot.py>')
import data

### Step 1 - (local only) stage the data

If `$PLOT_RUN_LOC = local`, copy the run outputs into a `datacopy/` folder **preserving
the `edmft-T<T>/GF/` subtree** so the `GF` tag still fires, and point the mission at
`datacopy/`. If remote, skip this and point the mission straight at the run dir.

### Step 2 - Build the `datacol` (append -> set -> pull)

Create the mission *inside* the `datacol`, then configure it before pulling. `pull()` only
fills a mission's `.data` - it does **not** set the member variables (`dir`, `title`,
`comment`, `target_list`, `label_list`, `type`); `set(...)` does, so `set()` **must precede**
`pull()`. `data.target[0]` = `['sig','cdos','gc1','dlt1']`; `type='edmft'` arms
`edmft_modifier`; `pull()` discovers those across the tree, incl. the `GF/` real-axis copies
(`GF/sig.inp` is the real-axis Sigma).

**Labels — generate from the run, don't hardcode 4f.** `data.label[0]` is a fixed 4f preset
(5/2, 7/2); `data.get` only swaps in generic `c0,c1,…` on a *length mismatch*, so a d-shell
whose column counts coincide (MnO) silently gets the wrong 4f names. `make_label(rundir)`
below reads the run's own orbital info instead — manifold names from the `*.gc1` header,
shell letter from `*.cdos`'s `L=` — and is passed to `set()` in place of `data.label[0]`.

In [ ]:
import glob, re
def make_label(rundir):
    # read the run's own orbital info -> label dict (mirrors the 4f preset shape in data.label[0])
    gc1  = sorted(glob.glob(rundir + '/**/*.gc1',  recursive=True))[0]   # manifold names in its header
    cdos = sorted(glob.glob(rundir + '/**/*.cdos', recursive=True))[0]   # correlated-shell L= in its header
    with open(gc1)  as f: manifolds = f.readline().replace('#','').split()[1:]   # e.g. ['eg','t2g'] or ['5/2','7/2']
    with open(cdos) as f: shell = {0:'s',1:'p',2:'d',3:'f'}[int(re.search(r'L=\s*(\d+)', f.readline()).group(1))]
    cols = ['w']
    for m in manifolds: cols += [m + '-r', m + '-i']
    return {'sig': cols, 'gc1': cols, 'dlt1': cols, 'cdos': ['w', 'total', shell]}

rundir = '<parent-of-edmft-T*-or-datacopy>'
col = data.datacol()
col.mission_list.append(data.mission())                    # one mission for the whole study
col.mission_list[0].set([rundir, '<CASE>', '<comment>'],
                         data.target[0], make_label(rundir), 'edmft')   # dynamic labels, not data.label[0]
col.mission_list[0].pull()                                 # walks all T-points; fills .data
col.save_pkl('<CASE>_data.pkl')

# Append a 2nd mission() (its own set/pull) only for a genuinely separate study, then pickle.

### Step 3 - Pickle, then comment out

`save_pkl` above wrote the state. Now **comment the build cell out** (wrap it in
`'''...'''`) so re-running won't re-walk or duplicate - the pkl is the record. When a new
batch has overwritten the folder, un-comment the build, `pull()` again (it appends the new
batch's results onto the mission), and `save_pkl`.

Verify the round-trip - reload the pickle and inspect what was collected (each line = one
file, with its `key_tag` and `c_num`):

In [ ]:
col = data.load_dcl('<CASE>_data.pkl')
col.mission_list[0].show()

### Append later (don't rebuild)

Reload the pickle, append a new mission, save again.

In [ ]:
'''
col = data.load_dcl('<CASE>_data.pkl')
col.mission_list.append(data.mission())
col.mission_list[-1].set(['<another-rundir>', '<CASE2>', '<comment>'],
                         data.target[0], make_label('<another-rundir>'), 'edmft')
col.mission_list[-1].pull()
col.save_pkl('<CASE>_data.pkl')
'''